# Train ASL classifier from published derived cache

Dùng sau khi `01_audit_cache_publish.ipynb` đã publish cache. Notebook xác minh SHA-256 raw archive rồi tải Parquet và hand crops; không chạy lại audit hay MediaPipe.

In [ ]:
%pip -q install 'huggingface-hub>=0.25' pandas pyarrow scikit-learn matplotlib seaborn
# TensorFlow GPU đã có sẵn trên Colab.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, zipfile
import pandas as pd, tensorflow as tf
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns

REPO_ID = 'hnam25/asl-hand-gesture-images'; RAW_ARCHIVE = 'ASL_HG_36000/ASL_Raw_Images.zip'; CACHE_PREFIX = 'derived/mediapipe-hand-landmarker-v1'
ROOT = Path('/content/asl-cache-train') if Path('/content').exists() else Path.cwd()/'asl-cache-train'
HF_ROOT, PROCESSED, OUTPUTS = ROOT/'hf', ROOT/'data/processed', ROOT/'outputs'
for directory in (HF_ROOT, PROCESSED, OUTPUTS/'models', OUTPUTS/'metrics', OUTPUTS/'figures', OUTPUTS/'logs', OUTPUTS/'metadata'): directory.mkdir(parents=True, exist_ok=True)
CLASSES = [str(i) for i in range(10)] + [chr(i) for i in range(ord('A'), ord('Z')+1)]
SEED, IMAGE_SIZE, BATCH_SIZE, EPOCHS = 42, 224, 32, 20
tf.keras.utils.set_random_seed(SEED)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(8*1024*1024), b''): digest.update(chunk)
    return digest.hexdigest()

snapshot_download(repo_id=REPO_ID, repo_type='dataset', local_dir=HF_ROOT, allow_patterns=[RAW_ARCHIVE, f'{CACHE_PREFIX}/**'])
raw_zip = next(HF_ROOT.rglob('ASL_Raw_Images.zip')); cache_root = HF_ROOT/CACHE_PREFIX
manifest = json.loads((cache_root/'cache_manifest.json').read_text())
if sha256_file(raw_zip) != manifest['raw_archive_sha256']: raise RuntimeError('Raw archive fingerprint differs from published cache; rebuild cache with notebook 01.')
with zipfile.ZipFile(cache_root/'processed_hand_crops.zip') as archive: archive.extractall(PROCESSED)
audit = pd.read_parquet(cache_root/'audit.parquet'); segmentation = pd.read_parquet(cache_root/'segmentation_manifest.parquet')
print('Cache verified:', manifest['raw_archive_sha256'], '| crops:', len(segmentation))

In [ ]:
# Deduplicate exact raw-image copies, then recreate the locked 80/10/10 split.
usable = segmentation[segmentation.status == 'ok'].merge(audit[audit.status == 'ok'][['source_relative', 'label', 'sha256']], how='left', on='source_relative', suffixes=('', '_audit'), validate='one_to_one')
if usable.sha256.isna().any() or (usable.label != usable.label_audit).any(): raise RuntimeError('Audit and segmentation manifests do not match.')
if (usable.groupby('sha256').label.nunique() > 1).any(): raise RuntimeError('The same raw image hash has conflicting labels.')
usable = usable.sort_values(['sha256', 'source_relative'], kind='stable').reset_index(drop=True)
usable['duplicate_group_size'] = usable.groupby('sha256').sha256.transform('size')
usable['canonical_source_relative'] = usable.groupby('sha256').source_relative.transform('first')
usable['is_canonical'] = usable.source_relative.eq(usable.canonical_source_relative)
usable['processed_path'] = usable.processed_relative.map(lambda value: str(PROCESSED/value))
if not usable.processed_path.map(lambda value: Path(value).is_file()).all(): raise RuntimeError('Processed crop archive is incomplete.')
usable[['label', 'source_relative', 'processed_relative', 'sha256', 'duplicate_group_size', 'canonical_source_relative', 'is_canonical']].to_csv(OUTPUTS/'metadata'/'deduplication_manifest.csv', index=False)
before_dedup = len(usable); usable = usable[usable.is_canonical].copy()
train, holdout = train_test_split(usable, train_size=.8, stratify=usable.label, random_state=SEED)
validation, test = train_test_split(holdout, train_size=.5, stratify=holdout.label, random_state=SEED)
if set(train.sha256) & set(validation.sha256) or set(train.sha256) & set(test.sha256) or set(validation.sha256) & set(test.sha256): raise RuntimeError('Hash leakage detected.')
(OUTPUTS/'metadata'/'split_manifest.json').write_text(json.dumps({'policy': 'one representative per exact SHA-256', 'before_deduplication': before_dedup, 'after_deduplication': len(usable), 'removed_exact_duplicates': before_dedup-len(usable), 'seed': SEED}, indent=2))
label_index = {label: index for index, label in enumerate(CLASSES)}
def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((frame.processed_path.values, frame.label.map(label_index).values))
    if training: ds = ds.shuffle(len(frame), seed=SEED)
    def load(path, label):
        image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False); image.set_shape([None, None, 3])
        return tf.image.resize(tf.cast(image, tf.float32), (IMAGE_SIZE, IMAGE_SIZE)), tf.one_hot(label, len(CLASSES))
    return ds.map(load, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds, validation_ds, test_ds = make_dataset(train, True), make_dataset(validation), make_dataset(test)
print({'before_dedup': before_dedup, 'after_dedup': len(usable), 'train': len(train), 'validation': len(validation), 'test': len(test)})


In [ ]:
# Train MobileNetV2 baseline from cached inputs.
backbone = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3)); backbone.trainable = False
inputs = tf.keras.Input((IMAGE_SIZE, IMAGE_SIZE, 3)); features = backbone(tf.keras.applications.mobilenet_v2.preprocess_input(inputs), training=False)
features = tf.keras.layers.GlobalAveragePooling2D()(features); features = tf.keras.layers.Dropout(.2)(features)
model = tf.keras.Model(inputs, tf.keras.layers.Dense(len(CLASSES), activation='softmax')(features))
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
checkpoint = OUTPUTS/'models/baseline_mobilenetv2.keras'
history = model.fit(train_ds, validation_data=validation_ds, epochs=EPOCHS, verbose=2, callbacks=[tf.keras.callbacks.ModelCheckpoint(checkpoint, monitor='val_accuracy', save_best_only=True), tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=2, factor=.2), tf.keras.callbacks.CSVLogger(OUTPUTS/'logs/training_history.csv')])

In [ ]:
# Evaluate overall and O/0-specific metrics.
best = tf.keras.models.load_model(checkpoint); probability = best.predict(test_ds, verbose=1); predicted = probability.argmax(1); truth = test.label.map(label_index).to_numpy()
report = classification_report(truth, predicted, labels=range(36), target_names=CLASSES, output_dict=True, zero_division=0); matrix = confusion_matrix(truth, predicted, labels=range(36))
pd.DataFrame(report).T.to_csv(OUTPUTS/'metrics/classification_report.csv'); pd.DataFrame(matrix, index=CLASSES, columns=CLASSES).to_csv(OUTPUTS/'metrics/confusion_matrix.csv')
o, zero = label_index['O'], label_index['0']; summary = {'test_accuracy': float(accuracy_score(truth, predicted)), 'O_recall': report['O']['recall'], '0_recall': report['0']['recall'], 'O_to_0': int(matrix[o, zero]), '0_to_O': int(matrix[zero, o])}
(OUTPUTS/'metrics/o_zero_analysis.json').write_text(json.dumps(summary, indent=2)); print(summary)
plt.figure(figsize=(16,13)); sns.heatmap(matrix, cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout(); plt.savefig(OUTPUTS/'figures/confusion_matrix.png', dpi=180); plt.close()